# CellTrack Studio Official Metric for exact d3ac1a 0.900

This notebook scores the **final postprocessed d3ac1a batch submission CSVs** with CellTrack Studio's vendored official Biohub metric. It computes exact edge matching, node-count adjustment, connected-component division scoring, and the final combined score.

The score is an in-sample sparse-GT diagnostic because the learned model used competition training data. It is useful for controlled method comparison and failure analysis, not as an unbiased Kaggle leaderboard estimate.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development')
BASE_DIR = PROJECT_DIR / 'data'
GT_DIR = BASE_DIR / 'train'
ROOT_REPORT_DIR = PROJECT_DIR / 'reports'
D3_REPORT_DIR = ROOT_REPORT_DIR / 'd3ac1a_0900_full199'
OUTPUT_DIR = D3_REPORT_DIR / 'celltrack_studio_official_metric'
SUPPORT_DIR = PROJECT_DIR / 'artifacts/biohub-tracking-support-pack-50ep-v1'
WHEEL_DIR = SUPPORT_DIR / 'wheels'
STUDIO_DIR = Path('/content/celltrack-studio')
STUDIO_COMMIT = '6daa1a50e1ae8addba5b6d35cb7d4fff323bcbc6'
BATCH_GLOB = 'd3ac1a_0900_full199_exact_batch_*_local_validation_submission.csv'
EXPECTED_FULL199 = 199

for path in [GT_DIR, D3_REPORT_DIR, WHEEL_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('GT_DIR:', GT_DIR)
print('d3ac1a reports:', D3_REPORT_DIR)
print('official metric output:', OUTPUT_DIR)
print('available exact batch CSVs:', len(list(D3_REPORT_DIR.glob(BATCH_GLOB))))

## Install the pinned CellTrack Studio metric

The GUI dependencies are intentionally not installed. Colab uses only the headless official metric and graph conversion path.

In [ ]:
def run(cmd, **kwargs):
    print(' '.join(map(str, cmd)))
    return subprocess.run([str(x) for x in cmd], check=True, **kwargs)

if not (STUDIO_DIR / '.git').exists():
    run(['git', 'clone', 'https://github.com/tom99763/celltrack-studio.git', STUDIO_DIR])
run(['git', '-C', STUDIO_DIR, 'fetch', '--depth', '1', 'origin', STUDIO_COMMIT])
run(['git', '-C', STUDIO_DIR, 'checkout', '--detach', STUDIO_COMMIT])

required = [
    'polars', 'tracksdata', 'zarr', 'geff', 'geff_spec', 'rustworkx',
    'numcodecs', 'donfig', 'bidict', 'deprecated', 'imagecodecs',
]
probe = subprocess.run(
    [sys.executable, '-c', 'import polars, tracksdata, zarr, geff, scipy'],
    text=True, capture_output=True,
)
if probe.returncode != 0:
    print('Refreshing scoring dependencies from the existing offline wheel pack.')
    run([
        sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
        '--force-reinstall', '--find-links', WHEEL_DIR, *required,
    ])
run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', STUDIO_DIR])

# Remove stale imports if this cell is rerun after a package refresh.
for prefix in ('polars', 'tracksdata', 'zarr', 'geff', 'celltrack_studio'):
    for name in list(sys.modules):
        if name == prefix or name.startswith(prefix + '.'):
            sys.modules.pop(name, None)
sys.path.insert(0, str(STUDIO_DIR))

import hashlib
metric_path = STUDIO_DIR / 'celltrack_studio/_vendor/tracking_cellmot/metrics.py'
division_path = STUDIO_DIR / 'celltrack_studio/_vendor/tracking_cellmot/division_metrics.py'
metric_sha = hashlib.sha256(metric_path.read_bytes()).hexdigest().upper()
division_sha = hashlib.sha256(division_path.read_bytes()).hexdigest().upper()
assert metric_sha == 'F7305D3D0A571479741DA1FB38A85A3CB357B07D5B46B8E06FF6F6B37AC784F7'
assert division_sha == '60FAA6BF682AF282C546E0AA203847EB7E0843297ED112FFDE04B1AAE7C3C71F'
print('Official metric source hashes verified.')

## Build the exact scoring inventory

Each final batch CSV is mapped to its datasets. Raw-before-clip files are excluded by the exact filename pattern. Duplicate datasets are rejected.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import tracksdata as td
import zarr
from geff import GeffMetadata
from tqdm.auto import tqdm
from IPython.display import display
from celltrack_studio._vendor.tracking_cellmot import metrics

batch_csvs = sorted(D3_REPORT_DIR.glob(BATCH_GLOB))
if not batch_csvs:
    raise FileNotFoundError(
        f'No exact d3ac1a postprocess batch CSVs found under {D3_REPORT_DIR}. '
        'Let the Full199 notebook reach postprocess+score, then rerun.'
    )

sample_to_csv = {}
inventory_rows = []
for csv_path in batch_csvs:
    ids = (
        pl.scan_csv(csv_path)
        .select(pl.col('dataset').cast(pl.String))
        .unique()
        .collect()['dataset']
        .to_list()
    )
    for sample_id in ids:
        if sample_id in sample_to_csv:
            raise RuntimeError(f'Duplicate final prediction for {sample_id}: {sample_to_csv[sample_id]} and {csv_path}')
        sample_to_csv[sample_id] = csv_path
    inventory_rows.append({'csv': csv_path.name, 'n_samples': len(ids)})

sample_ids = sorted(sample_to_csv)
missing_gt = [sid for sid in sample_ids if not (GT_DIR / f'{sid}.geff').exists()]
missing_image = [sid for sid in sample_ids if not (GT_DIR / f'{sid}.zarr').exists()]
print('batch CSVs:', len(batch_csvs))
print('final postprocessed samples available:', len(sample_ids), '/', EXPECTED_FULL199)
print('missing GT:', missing_gt)
print('missing image metadata:', missing_image)
display(pd.DataFrame(inventory_rows))
assert not missing_gt and not missing_image

def build_graph_from_submission_rows(group: pl.DataFrame) -> td.graph.InMemoryGraph:
    node_rows = group.filter(pl.col('row_type') == 'node')
    edge_rows = group.filter(pl.col('row_type') == 'edge')
    graph = td.graph.InMemoryGraph()
    for key in ('z', 'y', 'x'):
        graph.add_node_attr_key(key, pl.Float64, -999999.0)
    assigned = graph.bulk_add_nodes(
        node_rows.select(
            pl.col('t').cast(pl.Int64),
            pl.col('z').cast(pl.Float64),
            pl.col('y').cast(pl.Float64),
            pl.col('x').cast(pl.Float64),
        ).to_dicts()
    )
    id_map = dict(zip(node_rows['node_id'].to_list(), assigned, strict=True))
    if edge_rows.height:
        graph.bulk_add_edges([
            {'source_id': id_map[int(s)], 'target_id': id_map[int(t)]}
            for s, t in zip(edge_rows['source_id'], edge_rows['target_id'], strict=True)
        ])
    return graph

def load_graph(path: Path):
    result = td.graph.IndexedRXGraph.from_geff(path)
    return result[0] if isinstance(result, tuple) else result

def read_scale(sample_id: str):
    grp = zarr.open_group(str(GT_DIR / f'{sample_id}.zarr'), mode='r')
    attrs = dict(grp.attrs)
    try:
        transform = attrs['multiscales'][0]['datasets'][0]['coordinateTransformations'][0]
        if transform['type'] == 'scale':
            return tuple(float(v) for v in transform['scale'][-3:])
    except Exception:
        pass
    return (1.625, 0.40625, 0.40625)

def read_estimated_nodes(path: Path):
    try:
        meta = GeffMetadata.read(path)
        value = (meta.extra or {}).get('estimated_number_of_nodes')
        return float(value) if value is not None else float('nan')
    except Exception:
        return float('nan')

## Run resumable official scoring

A checkpoint CSV is rewritten after every sample. Rerunning the cell skips completed samples. Division evaluation can be substantially slower on division-heavy samples.

In [ ]:
PER_SAMPLE_PATH = OUTPUT_DIR / 'd3ac1a_official_metric_per_sample.csv'
ERROR_PATH = OUTPUT_DIR / 'd3ac1a_official_metric_errors.csv'
SUMMARY_PATH = OUTPUT_DIR / 'd3ac1a_official_metric_summary.csv'
BY_EMBRYO_PATH = OUTPUT_DIR / 'd3ac1a_official_metric_by_embryo.csv'
WORST_PATH = OUTPUT_DIR / 'd3ac1a_official_metric_worst40.csv'

if PER_SAMPLE_PATH.exists():
    result_rows = pd.read_csv(PER_SAMPLE_PATH).to_dict('records')
else:
    result_rows = []
if ERROR_PATH.exists():
    error_rows = pd.read_csv(ERROR_PATH).to_dict('records')
else:
    error_rows = []
done = {str(row['sample_id']) for row in result_rows}

bar = tqdm(sample_ids, desc='CellTrack official metric', unit='sample')
for sample_id in bar:
    if sample_id in done:
        bar.set_postfix(done=f'{len(done)}/{len(sample_ids)}', status='reused')
        continue
    started = time.time()
    try:
        csv_path = sample_to_csv[sample_id]
        group = (
            pl.scan_csv(csv_path)
            .filter(pl.col('dataset').cast(pl.String) == sample_id)
            .select(['row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id'])
            .collect()
        )
        pred_graph = build_graph_from_submission_rows(group)
        gt_path = GT_DIR / f'{sample_id}.geff'
        gt_graph = load_graph(gt_path)
        scale = read_scale(sample_id)
        n_estimated = read_estimated_nodes(gt_path)

        er = metrics.evaluate(pred_graph, gt_graph, scale=scale, max_distance=7.0)
        recall = metrics.node_recall(pred_graph, gt_graph) if pred_graph.num_nodes() else 0.0
        row = metrics.per_sample_metrics(er, n_estimated, recall)
        div_den = er.division_tp + er.division_fp + er.division_fn
        row.update({
            'sample_id': sample_id,
            'embryo_id': sample_id.split('_')[0],
            'scale_z': scale[0], 'scale_y': scale[1], 'scale_x': scale[2],
            'estimated_number_of_nodes': n_estimated,
            'division_jaccard': er.division_tp / div_den if div_den else np.nan,
            'source_csv': csv_path.name,
            'elapsed_seconds': time.time() - started,
        })
        result_rows.append(row)
        done.add(sample_id)
        pd.DataFrame(result_rows).sort_values('sample_id').to_csv(PER_SAMPLE_PATH, index=False)
        bar.set_postfix(done=f'{len(done)}/{len(sample_ids)}', edge=f"{row['edge_jaccard']:.3f}", div=f"{row['division_jaccard']:.3f}")
    except Exception as exc:
        error_rows = [r for r in error_rows if str(r.get('sample_id')) != sample_id]
        error_rows.append({
            'sample_id': sample_id, 'error_type': type(exc).__name__,
            'error': str(exc), 'source_csv': str(sample_to_csv.get(sample_id, '')),
        })
        pd.DataFrame(error_rows).to_csv(ERROR_PATH, index=False)
        bar.set_postfix(done=f'{len(done)}/{len(sample_ids)}', status='ERROR')
        print(f'ERROR {sample_id}: {type(exc).__name__}: {exc}')

scores_df = pd.DataFrame(result_rows).sort_values('sample_id').reset_index(drop=True)
metric_rows = scores_df[list(metrics.METRIC_COLUMNS)].to_dict('records')
summary = metrics.summarise(metric_rows)
summary.update({
    'available_final_samples': len(sample_ids),
    'scored_samples': len(scores_df),
    'expected_full199': EXPECTED_FULL199,
    'complete_full199': len(scores_df) == EXPECTED_FULL199,
    'metric_commit': STUDIO_COMMIT,
})
summary_df = pd.DataFrame([summary])
summary_df.to_csv(SUMMARY_PATH, index=False)

by_embryo_df = scores_df.groupby('embryo_id').agg(
    n_samples=('sample_id', 'count'),
    mean_edge_jaccard=('edge_jaccard', 'mean'),
    mean_adj_edge_jaccard=('adj_edge_jaccard', 'mean'),
    mean_division_jaccard=('division_jaccard', 'mean'),
    mean_node_recall=('node_recall', 'mean'),
    edge_tp=('edge_tp', 'sum'), edge_fp=('edge_fp', 'sum'), edge_fn=('edge_fn', 'sum'),
    division_tp=('division_tp', 'sum'), division_fp=('division_fp', 'sum'), division_fn=('division_fn', 'sum'),
).reset_index()
by_embryo_df.to_csv(BY_EMBRYO_PATH, index=False)
worst_df = scores_df.sort_values(['adj_edge_jaccard', 'edge_jaccard']).head(40)
worst_df.to_csv(WORST_PATH, index=False)

print('OFFICIAL METRIC SUMMARY')
display(summary_df)
print('BY EMBRYO')
display(by_embryo_df)
print('WORST SAMPLES')
display(worst_df)
print('saved:', OUTPUT_DIR)
if len(scores_df) < EXPECTED_FULL199:
    print(f'PARTIAL RESULT: {len(scores_df)}/{EXPECTED_FULL199}. Rerun after more d3ac1a postprocess batches finish.')

## Interpretation

- `score = weighted adjusted edge Jaccard + 0.1 * micro division Jaccard`.
- Unlike the previous approximate scorer, edge FP validity follows sparse-GT annotation boundaries and divisions use the official connected-component lineage rule.
- Compare pipeline variants only when they cover the same sample IDs.
- A Full199 training score is diagnostic and optimistic; Kaggle remains the final generalization check.
- The napari desktop GUI is not launched in Colab. The exact headless metric is the useful Colab-equivalent component.